# Prototipo B4 — De datos sueltos a objetos que deciden

**Cómo se trabaja aquí (igual que en B1–B3):** 1) lee el texto corto, 2) PREDICE antes de ejecutar, 3) ejecuta, 4) cambia algo pequeño, 5) comprueba con `assert`.

**Pregunta que te acompaña todo el bloque:** ¿por qué usar una clase aquí en lugar de seguir añadiendo funciones?

**Objetivo:** al terminar decides *qué objeto debe asumir cada responsabilidad* dentro de SAMI-OOP. La sintaxis es el medio; la decisión de diseño es la meta.

**Tiempo orientativo:** 60–90 minutos de cuaderno. El proyecto SAMI-OOP es donde se consolida (resto del bloque).


## Parte 1 — El problema antes de la clase

Un producto como diccionario. Funciona… hasta que alguien deja los datos en un estado incoherente.

In [ ]:
# EJECUTA. Un producto con estado válido… de momento.
producto = {"nombre": "Licencia", "precio_base": 80.0}
print(producto)

# Cualquier parte del programa puede hacer esto:
producto["precio_base"] = -500.0
print(producto)

# El diccionario no se queja: no sabe qué es un precio válido.
assert producto["precio_base"] == -500.0


**PREDICE antes de ejecutar la celda siguiente:** si calculas el precio final con IVA sobre ese diccionario, ¿qué sale? ¿Tiene sentido un precio final negativo?

In [ ]:
# EJECUTA. La función vive fuera del dato y no lo protege.
def calcular_precio_final_externo(producto, iva):
    return round(producto["precio_base"] * (1 + iva), 2)

precio_final = calcular_precio_final_externo(producto, 0.21)
print("Precio final:", precio_final)

# Absurdo pero "correcto" para la función: el dato ya venía roto.
assert precio_final == -605.0


**DECIDE (escribe una frase en tu cuaderno):** ¿qué datos y qué comportamiento deberían vivir juntos para que un precio negativo no pueda colarse? Todavía no escribas ninguna clase.

## Parte 2 — Clase, objeto, instancia (mínimo)

**Clase** → la definición (el molde). **Objeto / instancia** → la cosa concreta creada a partir de ella.

Eso es todo. El resto se ve ejecutando.

In [ ]:
# EJECUTA. Un molde, dos cosas concretas.
class Producto:
    pass

teclado = Producto()
raton = Producto()
print(type(teclado))
print(teclado is raton)

assert isinstance(teclado, Producto)
assert isinstance(raton, Producto)
assert teclado is not raton


**PREDICE:** ¿qué guarda la variable si olvidas los paréntesis (`cosa = Producto`)? ¿Es eso un objeto? Ejecuta la celda siguiente y compruébalo.

In [ ]:
# EJECUTA. Sin paréntesis no nace ningún objeto.
cosa = Producto
print(cosa)
print(cosa is Producto)

assert cosa is Producto
assert not isinstance(cosa, Producto)


## Parte 3 — `__init__` y `self`

`__init__` se ejecuta solo al crear cada objeto y deja su estado listo. `self` es el objeto concreto que está ejecutando el método: `self.nombre` no es "el nombre en general", es *el nombre de este objeto*.

In [ ]:
# EJECUTA. Dos objetos, cada uno con sus valores.
class Producto:
    def __init__(self, nombre, precio_base):
        self.nombre = nombre
        self.precio_base = precio_base

teclado = Producto("Teclado", 50.0)
raton = Producto("Raton", 20.0)
print(teclado.nombre, teclado.precio_base)
print(raton.nombre, raton.precio_base)

# Cambiar uno no toca al otro: mismo molde, estados independientes.
teclado.precio_base = 45.0
assert teclado.precio_base == 45.0
assert raton.precio_base == 20.0


**PREDICE:** si ahora ejecutas `raton.precio_base = 99.0`, ¿cuánto vale `teclado.precio_base`? Cámbialo tú y comprueba que el otro no se mueve.

In [ ]:
# EJECUTA. El método usa el estado DEL objeto que lo llama.
class Producto:
    def __init__(self, nombre, precio_base):
        self.nombre = nombre
        self.precio_base = precio_base

    def ficha(self):
        return self.nombre + " cuesta " + str(self.precio_base)

teclado = Producto("Teclado", 45.0)
raton = Producto("Raton", 20.0)
print(teclado.ficha())
print(raton.ficha())

# Python pasa la instancia sola: tú NO escribes self al llamar.
assert teclado.ficha() == "Teclado cuesta 45.0"
assert raton.ficha() == "Raton cuesta 20.0"


### CHECKPOINT 1 — «Puedo crear objetos con estado independiente»

Si puedes explicar por qué `teclado` y `raton` comparten molde pero no valores, sigue. Si no, repite la Parte 3 cambiando precios antes de avanzar.

## Parte 4 — Métodos: lo que en B2 ya usabas sin saberlo

En B2 escribiste `texto.lower()` y `diccionario.get(...)`: un método es una acción asociada a un objeto, llamada con punto sobre ese objeto. Ahora creas los tuyos.

In [ ]:
# EJECUTA. El método lee el estado del propio objeto.
class Producto:
    def __init__(self, nombre, precio_base):
        self.nombre = nombre
        self.precio_base = precio_base

    def calcular_precio_final(self, tasa_iva):
        return round(self.precio_base * (1 + tasa_iva), 2)

teclado = Producto("Teclado", 45.0)
print(teclado.calcular_precio_final(0.21))

assert teclado.calcular_precio_final(0.21) == 54.45
assert teclado.calcular_precio_final(0.0) == 45.0


**ERROR ÚTIL — PREDICE:** este método olvida `self`. ¿Qué error da al llamarlo? ¿Quién intentaba pasar Python automáticamente?

In [ ]:
# EJECUTA. Lee el error: es la pista, no el enemigo.
class Rota:
    def mostrar():
        return "ok"

mensaje = ""
try:
    Rota().mostrar()
except TypeError as error:
    mensaje = str(error)
    print(type(error).__name__, ":", error)

assert mensaje != ""


## Parte 5 — Proteger el estado: no aceptar cualquier valor

Volvemos al precio negativo de la Parte 1. Idea: **el objeto no acepta cambios sin comprobarlos**. El precio solo se cambia con `set_precio`, que valida antes de guardar.

In [ ]:
# EJECUTA. Estado protegido por un método con reglas.
class ProductoSeguro:
    def __init__(self, nombre, precio_base):
        self.nombre = nombre
        self.set_precio(precio_base)

    def set_precio(self, precio):
        if not isinstance(precio, (int, float)):
            raise TypeError("El precio debe ser numerico")
        if precio <= 0:
            raise ValueError("El precio debe ser positivo")
        self.__precio_base = precio

    def get_precio(self):
        return self.__precio_base

cable = ProductoSeguro("Cable", 10.0)
cable.set_precio(12.0)
assert cable.get_precio() == 12.0

# Caso inválido: el objeto se niega y explica por qué.
rechazado = False
try:
    cable.set_precio(-5.0)
except ValueError as error:
    rechazado = True
    print("Rechazado:", error)
assert rechazado
assert cable.get_precio() == 12.0


In [ ]:
# EJECUTA. Matiz honesto: __ no es privacidad absoluta en Python.
print(hasattr(cable, "_ProductoSeguro__precio_base"))

# Existe un acceso interno con otro nombre. La protección real es
# diseñar UNA vía clara de cambio (set_precio) y validarla.
assert hasattr(cable, "_ProductoSeguro__precio_base")


## Parte 6 — Composición: ¿TIENE o ES?

Pregunta clave: **¿un objeto ES otro objeto, o TIENE otro objeto?** Una auditoría de mercado no ES un producto: **TIENE productos**. Eso es composición.

In [ ]:
# EJECUTA. La auditoría contiene productos (una lista de objetos).
class Producto:
    def __init__(self, nombre, precio_base):
        self.nombre = nombre
        self.precio_base = precio_base

class AuditoriaMercado:
    def __init__(self):
        self.productos = []

    def agregar_producto(self, producto):
        self.productos.append(producto)

auditoria = AuditoriaMercado()
auditoria.agregar_producto(Producto("Teclado", 50.0))
auditoria.agregar_producto(Producto("Raton", 20.0))
print(len(auditoria.productos))

assert len(auditoria.productos) == 2
assert auditoria.productos[0].nombre == "Teclado"


**DECIDE:** un pedido contiene productos. ¿El pedido ES un producto o TIENE productos? ¿Y una factura respecto a sus líneas? Escribe «TIENE-UN porque…» antes de seguir.

## Parte 7 — Herencia: solo cuando ES-UN

Un producto hardware **ES un** producto con algo extra (el peso). La subclase reutiliza lo del padre con `super().__init__(...)` — sin pasar `self`, que ya viaja solo.

In [ ]:
# EJECUTA. Una sola capa de herencia: especialización concreta.
class Producto:
    def __init__(self, nombre, precio_base):
        self.nombre = nombre
        self.precio_base = precio_base

class ProductoHardware(Producto):
    def __init__(self, nombre, precio_base, peso_kg):
        super().__init__(nombre, precio_base)
        self.peso_kg = peso_kg

teclado = ProductoHardware("Teclado", 80.0, 1.2)
print(isinstance(teclado, Producto))
print(teclado.nombre, teclado.precio_base, teclado.peso_kg)

assert isinstance(teclado, Producto)
assert teclado.nombre == "Teclado"
assert teclado.peso_kg == 1.2


**PREDICE:** ¿qué atributo falta si quitas la línea `super().__init__(...)`? Ejecuta y lee el `AttributeError`.

In [ ]:
# EJECUTA. Sin super(), lo heredado nunca se inicializa.
class SinSuper(Producto):
    def __init__(self, nombre):
        self.nombre = nombre

incompleto = SinSuper("Cable")
falta = False
try:
    print(incompleto.precio_base)
except AttributeError as error:
    falta = True
    print("AttributeError:", error)
assert falta


## Parte 8 — Sobrescritura y polimorfismo

Sobrescribir = redefinir en la hija un método del padre. Polimorfismo = **pedir la misma acción a objetos distintos y que cada uno responda con su regla**. No memorices la definición: mírala funcionar.

In [ ]:
# EJECUTA. Mismo método, reglas distintas por tipo de producto.
class Producto:
    def __init__(self, nombre, precio_base):
        self.nombre = nombre
        self.precio_base = precio_base

    def calcular_precio_final(self, tasa_iva):
        return round(self.precio_base * (1 + tasa_iva), 2)

class ProductoLicencia(Producto):
    def calcular_precio_final(self, tasa_iva):
        con_descuento = self.precio_base * 0.95
        return round(con_descuento * (1 + tasa_iva), 2)

class ProductoHardware(Producto):
    def __init__(self, nombre, precio_base, peso_kg):
        super().__init__(nombre, precio_base)
        self.peso_kg = peso_kg

    def calcular_precio_final(self, tasa_iva):
        base = super().calcular_precio_final(tasa_iva)
        if self.peso_kg > 5:
            base += self.peso_kg * 2.0
        return round(base, 2)

licencia = ProductoLicencia("Suite", 140.0)
hardware = ProductoHardware("Teclado", 80.0, 1.2)
print(licencia.calcular_precio_final(0.21))
print(hardware.calcular_precio_final(0.21))

assert licencia.calcular_precio_final(0.21) == 160.93
assert hardware.calcular_precio_final(0.21) == 96.8


In [ ]:
# EJECUTA. Una sola llamada para una lista mixta: cada objeto aplica su regla.
productos = [licencia, hardware]
total = 0.0
for producto in productos:
    final = producto.calcular_precio_final(0.21)
    print(producto.nombre, final)
    total += final
print("Total:", round(total, 2))

assert round(total, 2) == 257.73


### CHECKPOINT 2 — «Sé distinguir TIENE-UN de ES-UN y puedo explicar mi elección»

Explica en voz alta: ¿`AuditoriaMercado` TIENE o ES productos? ¿`ProductoHardware` TIENE o ES un producto? Si dudas, vuelve a las Partes 6–7.

**Ojo para la Parte 10:** aquí todas las clases aplican el IVA. En el SAMI-OOP real hay una sorpresa con la licencia. Guárdate la pregunta: ¿todas las clases tratan `tasa_iva` igual?

## Parte 9 — Decidir si necesito una clase (la parte esencial)

No todo necesita una clase. Para cada caso escribe **«Elijo ___ porque…»** en tu cuaderno ANTES de mirar las soluciones. Opciones: `dict` · `función` · `clase` · `composición` · `herencia`.

1. **Configuración simple** (umbral de alerta + IVA): solo guardar dos valores que se leen al arrancar.
2. **Producto con reglas propias**: rechazar precios negativos y calcular su precio final.
3. **Pedido que contiene productos**: agrupar varios productos y sumar el total.
4. **Producto especializado**: una licencia es un producto con clave de activación y descuento.
5. **Cálculo aislado sin estado**: convertir euros a dólares con una tasa dada.

In [ ]:
# SOLUCIONES. Míralas SOLO después de escribir tus 5 frases.
soluciones = {
    1: "dict: guardar dos valores sin reglas ni comportamiento no necesita clase",
    2: "clase: estado (precio) + reglas (validar) + comportamiento (calcular) viven juntos",
    3: "composicion: el pedido TIENE productos, no ES un producto",
    4: "herencia: la licencia ES UN producto con datos y reglas extra",
    5: "funcion: un calculo sin estado que recordar no necesita objeto",
}
for numero, criterio in soluciones.items():
    print(numero, "->", criterio)

assert soluciones[3].startswith("composicion")
assert soluciones[4].startswith("herencia")
assert soluciones[5].startswith("funcion")


## Parte 10 — SAMI-OOP: el proyecto real

| | B3 SAMI-Lite | B4 SAMI-OOP |
|---|---|---|
| Organización | funciones + módulos | objetos + responsabilidades |
| Precio inválido | cada función valida por su cuenta | el objeto se protege solo |
| Tipos de producto | `if tipo == ...` repartidos | cada subclase calcula lo suyo |
| Productos agrupados | listas sueltas | `AuditoriaMercado` los contiene |
| Tu autonomía | localizas QUÉ función cambiar | decides QUÉ CLASE asume cada cambio |

Abre `../proyecto-sami-oop/` en VS Code. Las celdas siguientes trabajan con el código real.

### ORIENTARSE (hazlo en VS Code, 10 min)

1. ¿Cuál es el archivo de entrada y qué función arranca el programa?
2. Lista las clases de `analizador.py` y escribe la responsabilidad de cada una en una frase.
3. ¿Qué objetos crea `main.py` y cuáles colaboran entre sí?
4. ¿Dónde hay composición? ¿Dónde hay herencia? Justifica con «TIENE-UN / ES-UN porque…».

In [ ]:
# EJECUTA desde la carpeta cuaderno. Sigue el flujo real:
# main -> AuditoriaMercado -> metodo del producto -> resultado.
import os
import sys
for candidata in ("../proyecto-sami-oop", "pruebas-b4-pathway/proyecto-sami-oop"):
    if os.path.isdir(candidata):
        sys.path.insert(0, candidata)
        break

from analizador import AuditoriaMercado, ProductoHardware, ProductoLicencia

auditoria = AuditoriaMercado(umbral_alerta=100.0)
auditoria.agregar_producto(ProductoHardware("Teclado", 80.0, 1.2))
auditoria.agregar_producto(ProductoLicencia("Suite", 140.0, "ABC-123"))
filas, total = auditoria.generar_reporte(0.21)
for fila in filas:
    print(fila)
print("Total:", total)

# COMPRENDER: la auditoría delega; cada producto calcula con SU regla.
assert total == 229.8
assert filas[0][3] == "PRECIO_NORMAL"
assert filas[1][3] == "ALERTA_PRECIO_ELEVADO"

# Estado independiente también entre auditorías.
otra = AuditoriaMercado(umbral_alerta=100.0)
assert otra.productos == []
assert len(auditoria.productos) == 2


### DECIDIR (sin receta)

Observa con lupa el reporte anterior y el método `calcular_precio_final` de cada clase del proyecto real.

**Pregunta 1:** ¿todas las clases tratan el parámetro `tasa_iva` de la misma forma? ¿Qué clase rompe la regla común y por qué importa para el total?

**Pregunta 2:** `ProductoHardware` acepta cualquier `peso_kg`, incluso negativo. ¿Qué clase debería responsabilizarse de impedirlo y por qué ella y no `AuditoriaMercado`?

In [ ]:
# COMPRUEBA. Tres casos sobre el proyecto real: normal, límite e inválido.
from analizador import AuditoriaMercado, Producto, ProductoHardware

# Caso normal: precio válido calcula y evalúa.
auditoria = AuditoriaMercado(umbral_alerta=100.0)
auditoria.agregar_producto(ProductoHardware("Teclado", 80.0, 1.2))
filas, total = auditoria.generar_reporte(0.21)
assert filas[0][2] == 96.8

# Caso límite: precio final justo en el umbral NO dispara alerta (es > estricto).
assert auditoria.evaluar_alerta(100.0) == "PRECIO_NORMAL"
assert auditoria.evaluar_alerta(100.01) == "ALERTA_PRECIO_ELEVADO"

# Caso inválido: el objeto rechaza nacer con precio negativo.
rechazado = False
try:
    Producto("Roto", -5.0)
except ValueError:
    rechazado = True
assert rechazado

# Caso inválido: la auditoría solo acepta productos.
rechazado = False
try:
    auditoria.agregar_producto("no es un producto")
except TypeError:
    rechazado = True
assert rechazado
print("normal + limite + invalido: OK")


### MODIFICAR + MINI-RETO (en `../proyecto-sami-oop/`, no en este cuaderno)

**Cambio acotado:** `ProductoHardware` debe rechazar `peso_kg` no positivo (usa el estilo de `set_precio`: excepción clara). Compruébalo con caso normal (`1.2` aceptado), límite (`0` rechazado) e inválido (`-3` rechazado).

**MINI-RETO (objetivo + restricciones, sin receta):** unifica el significado de `tasa_iva` en las tres clases de `analizador.py` sin cambiar ninguna firma. Restricciones: la licencia conserva su descuento del 5% cuando el precio supera 100; el hardware conserva su recargo por peso; la lista mixta del reporte debe seguir calculándose con la misma llamada. Evidencia: muestra el reporte antes y después con los mismos datos.

### Autonomía: cómo sube el listón respecto a B3

- **B1:** sigues instrucciones. **B2:** eliges colección. **B3:** localizas QUÉ función modificar. **B4:** decides QUÉ CLASE asume cada responsabilidad y lo justificas.

La dificultad sube por criterio, no por sintaxis rara. Si tu justificación usa «TIENE-UN / ES-UN porque…», vas bien.

**Hueco antes de B5:** aquí los datos viven en memoria y en CSV; todavía no hay decisiones sobre errores de ficheros ausentes ni validación de formatos externos. Eso viene después.